# 3DXRD indexing notebook - Friedel pairs, windows and cuts from the data
__Written by Haixing Fang, Jon Wright and James Ball__
__Date: 20/11/2026__

For box-beam data over a 360 degree rotation. A grain diffracts $\mathbf{g}$ and $-\mathbf{g}$, usually near $\omega$
and $\omega+180$. With $\mathbf{g}_s(\mathbf{t})$ the sample-frame g-vector of a spot computed for a grain at
$\mathbf{t}$, the two spots of a pair satisfy $\mathbf{g}_{s1}(\mathbf{t}) + \mathbf{g}_{s2}(\mathbf{t}) = 0$. So a pair gives

* a g-vector that does not depend on where the grain is: the rings lose the broadening from grain positions, and
  the whole sample is indexed in one go, without a grid search;
* a constraint on the grain position: the pairs of one grain fix it.

Wedge and chi are taken into account (where $-\mathbf{g}$ diffracts is computed, not assumed at $\omega+180$).

Nothing below is a tolerance to tune by hand except the indexer's `hkl_tols_seq` and the refinement `makemap_tol_seq`:

| step | measured from the data |
|---|---|
| pair window | spread of each coordinate ($\omega$, $\eta$, $2\theta$, $\log I$) over the pairs; cut where pairs stop outnumbering partners found at a wrong $\omega$ |
| intensity floor | fraction of peaks with a partner (where the partner could be seen: detector, gaps, mask) against the chance fraction |
| ring window `dstol` | width of each ring in the pair geometry (Gaussian plus flat background); out to where the ring falls to the background |
| indexer `minpks` | the best score of the same search on scrambled peaks (each $\mathbf{g}$ turned about the rotation axis by a random angle) |
| grain positions | least squares on $\mathbf{g}_{s1}+\mathbf{g}_{s2}$ over the pairs of each grain |
| keeping a grain | its unclaimed peaks (all peaks) against random lattices on scrambled peaks, for about `max_false_grains` false grains |

**Several phases**: run once per phase. Each run adds a column `labels_<phase_str>` (grain id, -1 = not this phase)
to the 3D peaks; peaks claimed in another phase's column are not used (`exclude_phases`). The columns `labels`, `h`,
`k`, `l`, `drlv2`, ... belong to the phase run last.

In [ ]:
import os

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())

In [ ]:
# this cell is tagged with 'parameters'
# to view the tag, select the cell, then find the settings gear icon (right or left sidebar) and look for Cell Tags

# python environment stuff
IMAGED11_PATH = None  # means do not use git, otherwise "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = None  # None means guess, or you can specify a folder for the checkout

dset_path = ''

phase_str = 'Fe'

# path to parameters .json/.par
parfile = ''

# peaks for pairing and indexing: up to this d* (the intensity floor is measured below)
cf_strong_dsmax = 1.01

# rings, numbered as in the ring table below (they can also be picked there):
# orientations are made from peaks on rings_for_gen and scored on rings_for_scoring.
# Use rings clear of other phases.
rings_for_gen = [1, 3]
rings_for_scoring = [0, 1, 2, 3]

# multi-phase: peaks claimed by these phases (columns labels_<phase>) are not used
# None: every other phase that has a column; []: use all peaks
exclude_phases = None

# indexer: hkl tolerances, tried in turn, and the most grains per pair of rings and tolerance
hkl_tols_seq = [0.01, 0.02]
max_grains = 1000

# refinement (makemap); the last tolerance is also used to accept grains and assign peaks
symmetry = "cubic"
makemap_tol_seq = [0.02, 0.015, 0.01]

# expected number of false grains allowed
max_false_grains = 1

In [ ]:
if IMAGED11_PATH is not None:
    exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())
    PYTHONPATH=setup_ImageD11_from_git(CHECKOUT_PATH, IMAGED11_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
import os
import time
import numpy as np
import h5py
import ipywidgets
from IPython.display import display
from matplotlib import pyplot as plt

import ImageD11.cImageD11
import ImageD11.grain
import ImageD11.indexing
import ImageD11.columnfile
from ImageD11.sinograms import dataset
import fabio
from ImageD11.nbGui import nb_utils as utils
from ImageD11 import friedel_stats, grid_stats

%matplotlib widget

# Load data
## Dataset

In [ ]:
ds = ImageD11.sinograms.dataset.load(dset_path)
print(ds)

## Parameters
Specify the path to your parameter file.

You can optionally set up some default parameters for either an Eiger or Frelon detector like so:
```python
from ImageD11.parameters import AnalysisSchema
asc = AnalysisSchema.from_default(detector='eiger')  # or detector='frelon'
asc.save('./pars.json')
```
Please note in this case that you will still have to update the `geometry.par` values accordingly for your experiment.  
If you haven't already, you should run one of the calibration notebooks to determine these.

In [ ]:
ds.parfile = parfile
ds.save()

## Phases
If the parameter file was a json, we can access the unit cells via `ds.phases.unitcells`

In [ ]:
ds.phases = ds.get_phases_from_disk()
ds.phases.unitcells

In [ ]:
ucell = ds.phases.unitcells[phase_str]
print(ucell.lattice_parameters, ucell.spacegroup)

## Peaks
With several phases, the peaks claimed by the other phases (`labels_<phase> >= 0`) are left out: `cf_phase` holds the
rows of `cf_3d` this phase can use (`phase_rows`).

In [ ]:
cf_3d = ds.get_cf_3d_from_disk()
ds.update_colfile_pars(cf_3d, phase_name=phase_str)
label_col = 'labels_' + phase_str
others = [t[len('labels_'):] for t in cf_3d.titles if t.startswith('labels_') and t != label_col]
if exclude_phases is not None:
    others = [p for p in others if p in exclude_phases]
free = np.ones(cf_3d.nrows, bool)
for p in others:
    free &= cf_3d['labels_' + p] < 0
    print('phase', p, 'claims', (cf_3d['labels_' + p] >= 0).sum(), 'peaks')
phase_rows = np.flatnonzero(free)
cf_phase = cf_3d.copyrows(phase_rows)
print(cf_phase.nrows, 'of', cf_3d.nrows, 'peaks available for', phase_str)
print('omega from %.2f to %.2f, step %.4f' % (cf_phase.omega.min(), cf_phase.omega.max(), ds.ostep))

nproc = max(ImageD11.cImageD11.cores_available() - 1, 1)
omega_slop = ds.ostep / 2
oldparfile = phase_str + '.par'
ds.phases.to_old_pars_file(oldparfile, phase_str)
tmpfiles = [oldparfile]

# Friedel pairs
Pairs are searched among all peaks up to `cf_strong_dsmax` (no ring window and no intensity cut: which peaks are worth
using is measured below). Each peak's partner is the nearest peak to where its $-\mathbf{g}$ diffracts, in
$(\omega,\ \eta,\ 2\theta,\ \log I)$, with each coordinate in units of its spread over the pairs, measured on the pairs
themselves until it settles. Only mutual nearest partners are kept.

The same search $\delta$ = 7 and 13 degrees away in $\omega$ (far from real partners) finds only chance
neighbours. The cut keeps the pairs closer than the distance beyond which a wider window would add more chance
partners than real ones.

In [ ]:
cf_search = cf_phase.copyrows(cf_phase.ds <= cf_strong_dsmax)
t0 = time.time()
P = friedel_stats.pair_peaks(cf_search)
print('pair search on %d peaks (%.0f s)' % (cf_search.nrows, time.time() - t0))
print('spread of pairs:', ', '.join('%s %.3g' % (k, v) for k, v in zip(friedel_stats.NAMES, P['scale'])),
      '(degrees; log10 intensity)')

### Choose the cut
The slider starts at the cut measured above; moving it shows how many pairs a different cut keeps and how many
chance partners come with them. The next cell uses the slider's value.

In [ ]:
def cut_summary(cut):
    n_pairs = len(friedel_stats.pairs_within(P, cut)[0])
    n_obs = np.interp(cut, P['r'], P['n_obs'])
    n_null = np.interp(cut, P['r'], P['n_null'])
    return 'cut %.2f: %d pairs; %d peaks have a partner inside, about %d (%.0f%%) by chance' % (
        cut, n_pairs, n_obs, n_null, 100 * n_null / max(n_obs, 1))

fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
ax.plot(P['r'], P['n_obs'], label='partner where -g diffracts')
ax.plot(P['r'], P['n_null'], label='partner at a wrong omega (chance)')
ax.plot(P['r'], P['n_obs'] - P['n_null'], ':', label='difference (real partners)')
cut_line = ax.axvline(P['cut'], color='r', label='cut')
ax.set(xlabel='distance to the partner (spreads)', ylabel='peaks with a partner closer', title=cut_summary(P['cut']))
ax.legend()
plt.show()

cut_slider = ipywidgets.FloatSlider(value=P['cut'], min=0, max=float(P['r'][-1]), step=float(P['r'][1]),
                                    description='cut', readout_format='.2f', continuous_update=False)
def on_cut(change):
    cut_line.set_xdata([change['new'], change['new']])
    ax.set_title(cut_summary(change['new']))
    fig.canvas.draw_idle()
cut_slider.observe(on_cut, names='value')
display(cut_slider)

# Which peaks to use
A real diffraction spot has a partner unless the partner lands off the detector, in a module gap or under the mask
(`ds.maskfile`, 1 = masked; without one, the area the peaks cover). Those peaks are left out of this plot: a missing
partner says nothing there. For the others, the fraction with a partner is plotted against intensity, with the
fraction a chance search finds. Weak peaks are worth using as long as more of them have a real partner than a chance
one; the floor starts where that stops (going down from the best bin). The next cell uses the slider's value.

In [ ]:
pair_cut = cut_slider.value
mask = fabio.open(ds.maskfile).data if getattr(ds, 'maskfile', None) and os.path.exists(ds.maskfile) else None
observable, margin = friedel_stats.partner_observable(cf_search, P, mask)
print('partner predicted to within %.1f pixels; %d of %d peaks could have a partner (%s)' % (
    margin, observable.sum(), cf_search.nrows, 'mask ' + ds.maskfile if mask is not None else 'no mask: detector area from the peaks'))
logI = np.log10(np.maximum(cf_search.sum_intensity, 1))
centres, edges, f_real, f_chance, floor0 = friedel_stats.partner_fraction(P, logI, observable, cut=pair_cut)

def floor_summary(fl):
    keep = logI >= fl
    a, b = friedel_stats.pairs_within(P, pair_cut)
    return 'floor log10 I = %.2f: %d of %d peaks, %d pairs' % (fl, keep.sum(), len(keep), (keep[a] & keep[b]).sum())

fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
ax.plot(centres, f_real, 'o-', label='has a partner')
ax.plot(centres, f_chance, 's-', label='chance partner (wrong omega)')
floor_line = ax.axvline(floor0, color='r', label='floor')
ax.set(xlabel='log10 sum_intensity', ylabel='fraction of peaks (partner on the detector)', title=floor_summary(floor0))
ax.legend()
plt.show()

floor_slider = ipywidgets.FloatSlider(value=floor0, min=float(edges[0]), max=float(edges[-1]), step=0.01,
                                      description='floor', readout_format='.2f', continuous_update=False)
def on_floor(change):
    floor_line.set_xdata([change['new'], change['new']])
    ax.set_title(floor_summary(change['new']))
    fig.canvas.draw_idle()
floor_slider.observe(on_floor, names='value')
display(floor_slider)

## Rings in the pair geometry
Paired peaks get the pair g-vector, $(\mathbf{g}_{s1}-\mathbf{g}_{s2})/2$ at a $\mathbf{t}$ that satisfies the pair. For each ring: the offset of its
centre (the unit cell is slightly off if these grow with $d^*$), its width, and a window out to where the ring falls
to the background between rings.

The plot shows how far the peaks are from their ring in $d^*$, before pairing (computed for a grain at the origin, so
spread by the grain positions) and after pairing.

In [ ]:
intensity_floor = floor_slider.value
i1, i2 = friedel_stats.pairs_within(P, pair_cut)
keep = (logI[i1] >= intensity_floor) & (logI[i2] >= intensity_floor)
i1, i2 = i1[keep], i2[keep]
print(cut_summary(pair_cut), '; %d pairs above the floor' % len(i1))
cfp = friedel_stats.pair_colfile(cf_search, i1, i2)
ucell.makerings(cfp.ds.max())
ring_table = friedel_stats.ring_windows(cfp.ds, ucell.ringds, range(len(ucell.ringds)))
print('ring  hkl            d*       peaks  offset     sigma      backgr  window')
for r in sorted(ring_table):
    m, s, f, w = ring_table[r]
    n = (np.abs(cfp.ds - ucell.ringds[r] - m) < w).sum()
    print('%3d   %-12s  %.4f  %6d  %+.2e  %.2e  %.3f   %.2e' % (r, str(tuple(ucell.ringhkls[ucell.ringds[r]][0])), ucell.ringds[r], n, m, s, f, w))

def ring_offset(dstar):
    r = np.asarray(ucell.ringds)
    k = np.argmin(np.abs(dstar[:, None] - r[None, :]), axis=1)
    return dstar - r[k]
wmax = max(w[3] for w in ring_table.values())
fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
bins = np.linspace(-4 * wmax, 4 * wmax, 400)
ax.hist(ring_offset(cf_search.ds[logI >= intensity_floor]), bins=bins, histtype='step', label='peaks, grain at the origin')
ax.hist(ring_offset(cfp.ds), bins=bins, histtype='step', label='paired peaks, pair g-vector')
ax.set(xlabel=r'$d^* - d^*_\mathrm{ring}~(\AA^{-1})$', ylabel='peaks', yscale='log', title='Distance from the nearest ring')
ax.legend()
plt.show()

### Choose the rings and dstol
The ring selections start from the parameters; change them here if the table suggests other rings (a ring with a large
background or a wide window probably overlaps another phase). `dstol` starts at the widest window of the scoring rings
and follows the selection until it is moved. The next cell uses these values.

In [ ]:
labels = [('%d  %s  n=%d' % (r, str(tuple(ucell.ringhkls[ucell.ringds[r]][0])), (np.abs(cfp.ds - ucell.ringds[r] - ring_table[r][0]) < ring_table[r][3]).sum()), r)
          for r in sorted(ring_table)]
pick_gen = ipywidgets.SelectMultiple(options=labels, value=[r for r in rings_for_gen if r in ring_table],
                                     rows=min(len(labels), 12), description='generate')
pick_score = ipywidgets.SelectMultiple(options=labels, value=[r for r in rings_for_scoring if r in ring_table],
                                       rows=min(len(labels), 12), description='score')

pair_ring = np.argmin(np.abs(cfp.ds[:, None] - np.asarray(ucell.ringds)[None, :]), axis=1)
pair_offset = cfp.ds - np.asarray(ucell.ringds)[pair_ring]
def scoring_rings():
    return sorted(set(pick_score.value) | set(pick_gen.value))
def dstol_summary(d):
    on = np.isin(pair_ring, scoring_rings())
    return 'dstol %.2e: %d of %d paired peaks on rings %s are inside' % (
        d, (np.abs(pair_offset[on]) < d).sum(), on.sum(), scoring_rings())

d0 = max(ring_table[r][3] for r in scoring_rings())
fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
ax.hist(pair_offset, bins=np.linspace(-4 * wmax, 4 * wmax, 400), histtype='step', label='paired peaks')
dstol_lines = [ax.axvline(x, color='r') for x in (-d0, d0)]
ax.set(xlabel=r'$d^* - d^*_\mathrm{ring}~(\AA^{-1})$', ylabel='peaks', yscale='log', title=dstol_summary(d0))
ax.legend()
plt.show()

dstol_slider = ipywidgets.FloatSlider(value=d0, min=0, max=4 * wmax, step=wmax / 200, description='dstol',
                                      readout_format='.2e', continuous_update=False)
def on_dstol(change):
    for line, x in zip(dstol_lines, (-change['new'], change['new'])):
        line.set_xdata([x, x])
    ax.set_title(dstol_summary(change['new']))
    fig.canvas.draw_idle()
def on_rings(change):
    dstol_slider.value = max(ring_table[r][3] for r in scoring_rings())
    ax.set_title(dstol_summary(dstol_slider.value))
dstol_slider.observe(on_dstol, names='value')
pick_gen.observe(on_rings, names='value')
pick_score.observe(on_rings, names='value')
display(ipywidgets.VBox([ipywidgets.HBox([pick_gen, pick_score]), dstol_slider]))

# Indexing
For each `hkl_tol` and pair of generating rings, the trial orientations are scored and taken strongest first (an
orientation whose peaks are mostly taken is skipped). `minpks` is the best score that the same search reaches on
scrambled peaks, so chance alone is not expected to pass.

In [ ]:
rings_for_gen = sorted(pick_gen.value)
rings_for_scoring = scoring_rings()
dstol = dstol_slider.value
print('rings: generate', rings_for_gen, 'score', rings_for_scoring, '; dstol %.2e' % dstol)
t0 = time.time()
ImageD11.indexing.loglevel = 3  # warnings only
indexer, index_log = friedel_stats.index_pairs(
    cfp, ucell, [ucell.ringds[r] for r in rings_for_gen], [ucell.ringds[r] for r in rings_for_scoring], dstol,
    hkl_tols_seq, np.cos(np.radians(90 - ds.ostep)), max_grains)
print('hkl_tol  rings   minpks  trials  new grains')
for tol, r1, r2, minpks, ntrial, nnew in index_log:
    print('%.3f    %d,%d    %5d   %6d   %5d' % (tol, r1, r2, minpks, ntrial, nnew))
print('%d orientations (%.0f s)' % (len(indexer.ubis), time.time() - t0))

# Grain positions
A grain's position minimises $\sum |\mathbf{g}_{s1}(\mathbf{t}) + \mathbf{g}_{s2}(\mathbf{t})|^2$ over its pairs (pairs
further than three robust widths are dropped). Grains with fewer than three pairs are left out.

In [ ]:
pair_grain = np.full(len(i1), -1)
assigned = indexer.ga > 0
pair_grain[indexer.colfile['pair'][assigned]] = indexer.ga[assigned] - 1
T, rms, nlines = friedel_stats.fit_positions(cf_search, i1, i2, pair_grain, len(indexer.ubis))
ok = nlines >= 3
cands = [ImageD11.grain.grain(u, t) for u, t, k in zip(indexer.ubis, T, ok) if k]
print('%d grains with a position; rms |g_s1 + g_s2| of their pairs: median %.2e, 90%% %.2e' % (
    len(cands), np.median(rms[ok]), np.percentile(rms[ok], 90)))

fig, ax = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
ax[0].hist(rms[ok], bins=50)
ax[0].set(xlabel=r'rms $|g_{s1} + g_{s2}|$ ($\AA^{-1}$)', ylabel='grains')
ax[1].scatter(T[ok, 0], T[ok, 1], s=4, c=nlines[ok])
ax[1].set(xlabel='x', ylabel='y', aspect='equal', title='positions (colour: pairs used)')
plt.show()

# Refinement and acceptance
The grains are refined (makemap, `makemap_tol_seq[-1]`) on the peaks of all rings above the floor, repeated while the
positions still move. They are refined in parallel chunks, so peaks can be shared between chunks until the next step.

**Keeping a grain.** Each grain indexes the peaks within `makemap_tol_seq[-1]` of an integer hkl among **all** peaks of
the phase, at its own position. Strongest first, a grain is kept if more than $c$ of its peaks are not claimed by a
grain kept before, and then claims them; a duplicate has few unclaimed peaks. $c$ comes from random lattices at grain
positions, refined like a grain on scrambled peaks (each $\mathbf{g}$ turned about the rotation axis by a random angle,
so the rings and detector coverage stay, and no real grain can be found): with $N$ grains, about
$N\,P(u^\mathrm{rand} > c)$ false grains are expected, and $c$ starts at the smallest value for which that is at most
`max_false_grains`; the slider under the histogram moves it, and the next cell uses its value.

In [ ]:
def converge(grains, colfile, tol, tag, parallel, maxiter=6):
    """ makemap until the median position change stops halving.
    parallel: refined in chunks (peaks assigned within a chunk). Otherwise all grains together,
    and colfile gets the makemap columns. """
    prev = grains
    last = np.inf
    for i in range(maxiter):
        if parallel:
            cur, kept = grid_stats.refine_grains_parallel(oldparfile, colfile, prev, tol, omega_slop, symmetry, nproc)
        else:
            cur, kept = grid_stats.refine_grains(oldparfile, colfile, prev, tol, omega_slop, symmetry)
        shift = np.median([np.linalg.norm(np.subtract(prev[j].translation, g.translation)) for j, g in zip(kept, cur)])
        print('  makemap %s tol %g: median position change %.3g' % (tag, tol, shift))
        if shift > last / 2:
            break
        prev, last = cur, shift
    return [g for g in cur if 'no peaks' not in g.intensity_info]

t0 = time.time()
allrings_rows = np.flatnonzero(np.log10(np.maximum(cf_phase.sum_intensity, 1)) >= intensity_floor)
cands = converge(cands, cf_phase.copyrows(allrings_rows), makemap_tol_seq[-1], 'candidates', parallel=True)
print('%d grains refined (%.0f s)' % (len(cands), time.time() - t0))

In [ ]:
t0 = time.time()
final_tol = makemap_tol_seq[-1]
geom = grid_stats.PeakGeometry(cf_phase)
matched = grid_stats.match_grains(geom, cands, final_tol, nproc)
n_null = max(2000, int(np.ceil(5 * len(cands) / max_false_grains)))
rng = np.random.RandomState(0)
pos = [cands[i].translation for i in rng.choice(len(cands), int(np.ceil(n_null / 10)))]
nulls = grid_stats.null_matches(geom, pos, ucell.B, makemap_tol_seq, final_tol, 10, nproc, 0, scramble=True)
acc, claimed, cut, efalse, unclaimed, null_un = grid_stats.accept_grains(
    matched, nulls, np.zeros(cf_phase.nrows, bool), max_false_grains)
print('%d grains, %d random lattices (unclaimed median %d, max %d) -> cut %d for %g expected false grains (%.0f s)' % (
    len(cands), len(nulls), np.median(null_un), null_un.max(), cut, max_false_grains, time.time() - t0))

def accept_summary(c):
    kept, _ = grid_stats.greedy_accept(matched, c, np.zeros(cf_phase.nrows, bool))
    return 'cut %d: keep %d of %d grains, expected false %.2f' % (c, len(kept), len(cands), len(cands) * (null_un > c).mean())

fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
b = np.arange(0, max(unclaimed.max(), null_un.max()) + 10, 5)
ax.hist(unclaimed, bins=b, histtype='step', label='grains')
ax.hist(null_un, bins=b, histtype='step', weights=np.full(len(null_un), len(cands) / len(null_un)), label='random lattices (scaled)')
accept_line = ax.axvline(cut, c='r')
ax.set(yscale='log', xlabel='unclaimed peaks (hkl_tol %.4f)' % final_tol, ylabel='grains', title=accept_summary(cut))
ax.legend()
plt.show()

accept_slider = ipywidgets.IntSlider(value=cut, min=0, max=int(b[-1]), description='cut', continuous_update=False)
def on_accept(change):
    accept_line.set_xdata([change['new'], change['new']])
    ax.set_title(accept_summary(change['new']))
    fig.canvas.draw_idle()
accept_slider.observe(on_accept, names='value')
display(accept_slider)

In [ ]:
accept_cut = accept_slider.value
claimed = np.zeros(cf_phase.nrows, bool)
acc, _ = grid_stats.greedy_accept(matched, accept_cut, claimed)  # fills claimed
accepted = [cands[j] for j in sorted(acc)]
print('%s; %d of %d peaks claimed' % (accept_summary(accept_cut), claimed.sum(), len(claimed)))

# Final refinement on all peaks
All kept grains are refined together against all peaks of the phase (each peak goes to the grain that fits it best),
repeated while the positions still move.

In [ ]:
t0 = time.time()
threads = ImageD11.cImageD11.cimaged11_omp_get_max_threads()
ImageD11.cImageD11.cimaged11_omp_set_num_threads(nproc)  # makemap's peak assignment uses openmp
try:
    mm = cf_phase.copy()
    grains_final = converge([ImageD11.grain.grain(g.ubi, g.translation) for g in accepted], mm, final_tol, 'final', parallel=False)
finally:
    ImageD11.cImageD11.cimaged11_omp_set_num_threads(threads)
for g in grains_final:
    g.ref_unitcell = ucell
utils.get_rgbs_for_grains(grains_final)
ImageD11.grain.write_grain_file('final.map', grains_final)
mm.writefile('cf_phase.flt.new')
tmpfiles += ['final.map', 'cf_phase.flt.new']
print('%d final grains (%.0f s)' % (len(grains_final), time.time() - t0))

## Peak labels
The makemap columns (`labels`, `drlv2`, `h`, `k`, `l`, ...) are put back into `cf_3d`: rows of this phase from
makemap, other rows as unassigned (`labels` -1). `labels_<phase_str>` is a copy of `labels` that later runs for other
phases keep.

In [ ]:
per_grain = {'labels': -1, 'drlv2': 1, 'tth_per_grain': -1, 'eta_per_grain': 0, 'omegacalc_per_grain': 0,
             'h': 0, 'k': 0, 'l': 0, 'hr': 0, 'kr': 0, 'lr': 0}
for t in mm.titles:
    if t in per_grain or t not in cf_3d.titles:
        col = np.full(cf_3d.nrows, per_grain.get(t, 0), dtype=mm[t].dtype)
    else:
        col = cf_3d[t].copy()
    col[phase_rows] = mm[t]
    cf_3d.addcolumn(col, t)
cf_3d.addcolumn(cf_3d.labels.astype(np.int32), label_col)
print((cf_3d[label_col] >= 0).sum(), 'peaks assigned to', len(grains_final), phase_str, 'grains')

In [ ]:
utils.plot_grain_positions(grains_final, colour='npks', centre_plot=False, size_scaling=0.5)

In [ ]:
utils.plot_grain_positions(grains_final, colour='z', centre_plot=False, size_scaling=0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')
ax.hist([float(grain.npks) for grain in grains_final], bins=30)
ax.set(xlabel='Number of peaks per grain', ylabel='Count', title='Histogram of peaks per grain')
plt.show()

In [ ]:
utils.plot_grain_histograms('cf_phase.flt.new', 'final.map', oldparfile, omega_slop, tol=final_tol)

In [ ]:
unit_cell_lengths = np.array([np.sort(g.unitcell.copy()) for g in grains_final])
median_unit_cell = np.median(unit_cell_lengths, axis=0)
print("Median unit cell:", median_unit_cell)
fig, ax = plt.subplots(constrained_layout=True)
ax.plot(unit_cell_lengths[:,:3])
ax.hlines(median_unit_cell[:3], 0, len(unit_cell_lengths))
ax.set(xlabel="Grain ID", ylabel="Unit cell length")
plt.show()

# Export data
The peaks go to group `peaks` of the 3D peaks file (plus `labels_<phase_str>` as an integer column), the grains to
group `phase_str` of the grains file (replaced if it exists).

In [ ]:
ImageD11.columnfile.colfile_to_hdf(cf_3d, ds.col3dfile, name='peaks')
with h5py.File(ds.col3dfile, 'a') as hout:  # colfile_to_hdf writes columns it does not know as float
    grp = hout['peaks']
    if label_col in grp and grp[label_col].dtype != np.int32:
        del grp[label_col]
    if label_col in grp:
        grp[label_col][:] = cf_3d[label_col]
    else:
        grp.create_dataset(label_col, data=cf_3d[label_col].astype(np.int32))
if os.path.exists(ds.grainsfile):
    with h5py.File(ds.grainsfile, 'a') as hout:  # write_grain_file_h5 cannot overwrite a group
        if phase_str in hout:
            del hout[phase_str]
ds.save_grains_to_disk(grains_final, phase_name=phase_str)
ds.save()

In [ ]:
for path in set(tmpfiles):
    if os.path.exists(path):
        os.remove(path)